# TRex annotations to a YOLO dataset

This notebook finds multiple `videoname_annotations_yolo` folders and optional Roboflow exports. Roboflow projects are detected from the `roboflow:` block in their `data.yaml`, so the directory itself can have any name. T-REX frames are renamed to `videoname_frame_number`; already-renamed Roboflow files keep their names. Everything is pooled and `dataset-fixer` creates a new validated `train`/`val` dataset.

The default split keeps all frames from a video together. This avoids temporal leakage between training and validation. At least two video exports are needed for a video-grouped split.

## 1. Install the dependency

`dataset-fixer` requires Python 3.12 or newer. Check the kernel version before installing. It is currently installed directly from GitHub because no PyPI distribution is available.

In [ ]:
import sys

print(sys.version)
if sys.version_info < (3, 12):
    raise RuntimeError("Use a Jupyter kernel running Python 3.12 or newer.")

In [ ]:
%pip install "dataset-fixer @ git+https://github.com/mooch443/dataset-fixer.git"

## 2. Configure the conversion

Each T-REX export folder must contain exactly one `images/` and `labels/` pair. A detected Roboflow export may contain existing `train`, `valid`, and/or `test` image-label pairs; their old split membership is ignored and all frames are included in the new split. Set class names in numeric YOLO class-ID order. If `CLASS_NAMES` is `None`, the notebook prioritizes the complete class schema in a Roboflow `data.yaml`; otherwise it reads available dataset YAML files. If none contain names, names such as `class_0` are generated from the labels.

Set `OVERWRITE_OUTPUT = True` to replace an existing output directory. The notebook checks that the output is not the input directory or one of its parents before removing it.

### Choosing the train/validation split ratio

The split is controlled by three settings in the cell below:

- **`VALIDATION_FRACTION`** — the share of frames set aside for validation, as a number strictly between 0 and 1. For example `0.20` means a 80% train / 20% validation split. A common range is `0.10`–`0.30`; use a smaller fraction when you have very little data and want to keep more of it for training.
- **`SPLIT_BY_VIDEO`** — when `True` (recommended), every frame from the same source video (or Roboflow-inferred video) stays in the same split, so no video appears in both `train` and `val`. This avoids temporal leakage from near-duplicate consecutive frames. It requires at least two distinct videos; set it to `False` to split individual frames randomly instead (e.g. if you only have one video).
- **`RANDOM_SEED`** — a fixed seed so the split is reproducible: rerunning the notebook with the same inputs and seed always produces the same train/val assignment. Change it to get a different random split.

In [ ]:
from pathlib import Path

INPUT_ROOT = Path("//Users/aalbi/Seafile/moths/annotation-trex-exported/")
OUTPUT_DATASET = Path("/Users/aalbi/Seafile/moths/training-data/")
CLASS_NAMES = None  # Read data.yaml automatically, or set ["shark", "ray", "turtle"]
TASK = "detect"           # detect, segment, pose, or polo

# Split ratio settings — see the "Choosing the train/validation split ratio" note above.
VALIDATION_FRACTION = 0.20  # Fraction of frames reserved for validation, strictly between 0 and 1 (0.20 = 80/20 split)
RANDOM_SEED = 42            # Fixes the random split so it's reproducible across reruns
SPLIT_BY_VIDEO = True       # Keep all of a video's frames in one split to avoid train/val leakage; needs >= 2 videos

DEEP_DUPLICATE_CHECK = False
OVERWRITE_OUTPUT = True  # Set True to replace an existing output dataset

## 3. Converter functions

In [ ]:
import re
import shutil
import tempfile
from collections.abc import Iterable

import yaml
from dataset_fixer import Dataset

IMAGE_EXTENSIONS = {".bmp", ".jpeg", ".jpg", ".png", ".tif", ".tiff", ".webp"}
EXPORT_SUFFIX = "_annotations_yolo"


def video_id(folder: Path) -> str:
    """Derive a filesystem-safe video identifier from an export folder's name.

    Strips the '_annotations_yolo' suffix (if present) and replaces any
    character outside A-Z, a-z, 0-9, '.', '_', '-' with an underscore.
    """
    name = folder.name
    if name.lower().endswith(EXPORT_SUFFIX):
        name = name[:-len(EXPORT_SUFFIX)]
    cleaned = re.sub(r"[^A-Za-z0-9._-]+", "_", name).strip("_.")
    if not cleaned:
        raise ValueError(f"Cannot derive a video name from {folder}")
    return cleaned


def find_exports(root: Path) -> list[Path]:
    """Recursively find T-REX '*_annotations_yolo' export folders under root.

    root itself is included if it matches, so a single export can be passed
    directly as INPUT_ROOT.
    """
    root = root.expanduser().resolve()
    exports = sorted(
        path for path in root.rglob("*")
        if path.is_dir() and path.name.lower().endswith(EXPORT_SUFFIX)
    )
    if root.is_dir() and root.name.lower().endswith(EXPORT_SUFFIX):
        exports.insert(0, root)
    return exports


def find_roboflow_roots(root: Path) -> list[Path]:
    """Recursively find Roboflow export folders under root.

    Matches folders literally named 'roboflow', plus any folder whose
    data.yaml/dataset.yaml contains a 'roboflow:' metadata block, so a
    Roboflow download keeps working under its default project name.
    """
    root = root.expanduser().resolve()
    folders = [path for path in root.rglob("*") if path.is_dir() and path.name.casefold() == "roboflow"]
    if root.is_dir() and root.name.casefold() == "roboflow":
        folders.insert(0, root)

    # Standard Roboflow downloads are usually named after the project, not
    # 'roboflow'. Identify them by the metadata block in data.yaml.
    yaml_names = {"data.yaml", "data.yml", "dataset.yaml", "dataset.yml"}
    for path in root.rglob("*"):
        if not path.is_file() or path.name.lower() not in yaml_names:
            continue
        document = yaml.safe_load(path.read_text(encoding="utf-8")) or {}
        if isinstance(document, dict) and isinstance(document.get("roboflow"), dict):
            folders.append(path.parent)
    return sorted(set(folders))


def find_roboflow_pairs(root: Path) -> list[tuple[Path, Path]]:
    """Find every images/ + labels/ folder pair nested under a Roboflow root.

    Covers Roboflow exports that contain multiple pre-split pairs, e.g.
    train/images+train/labels, valid/images+valid/labels, test/images+test/labels.
    """
    pairs = [
        (images, images.parent / "labels")
        for images in root.rglob("images")
        if (images.parent / "labels").is_dir()
    ]
    return list(dict.fromkeys(pairs))


def class_names_from_yaml(root: Path) -> dict[int, str] | None:
    """Read class ID -> name mappings from dataset YAML files under root.

    A Roboflow project YAML is treated as authoritative over any other
    dataset YAML found (see the comment below), since T-REX exports may
    only list the class IDs present in that one video. Raises if multiple
    non-Roboflow YAML files disagree on names. Returns None if no YAML
    file defines names.
    """
    yaml_names = {"data.yaml", "data.yml", "dataset.yaml", "dataset.yml"}
    candidates = sorted(path for path in root.rglob("*") if path.is_file() and path.name.lower() in yaml_names)
    schemas = []
    for path in candidates:
        document = yaml.safe_load(path.read_text(encoding="utf-8")) or {}
        raw_names = document.get("names") if isinstance(document, dict) else None
        if isinstance(raw_names, list):
            names = {index: str(name).strip().strip("\"'") for index, name in enumerate(raw_names)}
        elif isinstance(raw_names, dict):
            try:
                names = {int(index): str(name).strip().strip("\"'") for index, name in raw_names.items()}
            except (TypeError, ValueError) as exc:
                raise ValueError(f"Invalid class IDs in {path}") from exc
        else:
            continue
        if names:
            schemas.append((path, names, isinstance(document.get("roboflow"), dict)))

    if not schemas:
        return None
    # A Roboflow project YAML describes the complete class schema. T-REX
    # exports may contain only the class IDs present in that one video and
    # placeholder names such as class_3, so they must not override it.
    authoritative = [(path, names) for path, names, is_roboflow in schemas if is_roboflow]
    considered = authoritative or [(path, names) for path, names, _ in schemas]
    expected_path, expected = considered[0]
    conflicts = [path for path, names in considered[1:] if names != expected]
    if conflicts:
        paths = "\n  - ".join(str(path) for path in [expected_path, *conflicts])
        raise ValueError(f"Conflicting class names in dataset YAML files:\n  - {paths}")
    print(f"Read class names from {expected_path}: {expected}")
    return expected


def locate_pair_dirs(export: Path) -> tuple[Path, Path]:
    """Find the single images/ + labels/ pair inside one T-REX export folder.

    Raises if the export contains zero or more than one such pair, since
    each T-REX export is expected to hold exactly one video's frames.
    """
    candidates = [(export / "images", export / "labels")]
    candidates.extend((images, images.parent / "labels") for images in export.rglob("images"))
    pairs = [(images, labels) for images, labels in candidates if images.is_dir() and labels.is_dir()]
    pairs = list(dict.fromkeys(pairs))
    if len(pairs) != 1:
        raise ValueError(
            f"{export} must contain exactly one images/ + labels/ pair; found {len(pairs)}"
        )
    return pairs[0]


def iter_images(folder: Path) -> Iterable[Path]:
    """Recursively yield supported image files under folder, in sorted order."""
    return sorted(path for path in folder.rglob("*") if path.suffix.lower() in IMAGE_EXTENSIONS)


def max_class_id(label_paths: Iterable[Path]) -> int:
    """Return the highest class ID referenced across the given YOLO label files.

    Used as a fallback to generate placeholder class names (class_0, class_1, ...)
    when no dataset YAML with real names is available. Returns -1 if no class
    IDs are found at all.
    """
    maximum = -1
    for label in label_paths:
        for line_number, raw in enumerate(label.read_text(encoding="utf-8").splitlines(), 1):
            fields = raw.split()
            if not fields:
                continue
            try:
                class_id = int(fields[0])
            except ValueError as exc:
                raise ValueError(
                    f"Invalid class ID in {label}:{line_number}: {fields[0]!r}"
                ) from exc
            maximum = max(maximum, class_id)
    return maximum


def stage_exports(exports: list[Path], stage: Path) -> tuple[int, list[Path]]:
    """Copy T-REX exports into a merged staging folder ahead of the split.

    Every image/label pair is renamed to '<video_id>_<frame>' so filenames
    stay globally unique once all videos are pooled together. Returns the
    number of images copied and the list of staged label paths.
    """
    image_root = stage / "train" / "images"
    label_root = stage / "train" / "labels"
    count = 0
    staged_labels = []
    seen_videos = set()

    for export in exports:
        source_id = video_id(export)
        if source_id.casefold() in seen_videos:
            raise ValueError(f"Duplicate derived video name {source_id!r}")
        seen_videos.add(source_id.casefold())
        images_dir, labels_dir = locate_pair_dirs(export)

        for image in iter_images(images_dir):
            relative = image.relative_to(images_dir)
            frame = "_".join(relative.with_suffix("").parts)
            destination_stem = f"{source_id}_{frame}"
            destination_image = image_root / f"{destination_stem}{image.suffix.lower()}"
            destination_label = label_root / f"{destination_stem}.txt"
            if destination_image.exists() or destination_label.exists():
                raise ValueError(f"Output filename collision: {destination_stem}")

            destination_image.parent.mkdir(parents=True, exist_ok=True)
            destination_label.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(image, destination_image)

            source_label = labels_dir / relative.with_suffix(".txt")
            if source_label.is_file():
                shutil.copy2(source_label, destination_label)
            else:
                destination_label.write_text("", encoding="utf-8")
            staged_labels.append(destination_label)
            count += 1

    if count == 0:
        raise ValueError("The annotation exports contain no supported images")
    return count, staged_labels


def roboflow_video_id(filename: str) -> str:
    """Guess the source video name from a Roboflow-exported filename.

    Handles plain frame names like video_000123.jpg as well as Roboflow's
    own naming scheme, e.g. video-000123_jpg.rf.<hash>.jpg. The Roboflow
    hash suffix and the trailing frame number are both stripped, leaving
    just the video name. Used to group frames by video for SPLIT_BY_VIDEO.
    """
    stem = Path(filename).stem
    stem = re.sub(r"_jpg\.rf\.[^.]+$", "", stem, flags=re.IGNORECASE)
    match = re.match(r"^(.*?)[_-](?:frame[_-]?)?\d+$", stem, flags=re.IGNORECASE)
    return match.group(1) if match else stem


def stage_roboflow(roots: list[Path], stage: Path) -> tuple[int, list[Path]]:
    """Copy Roboflow exports into the merged staging folder ahead of the split.

    Unlike stage_exports, original filenames are kept as-is (Roboflow names
    are already unique and frame-derived). Every images/+labels/ pair found
    under each root — including pre-existing train/valid/test pairs — is
    pooled together, since old split membership is discarded. Returns the
    number of images copied and the list of staged label paths.
    """
    image_root = stage / "train" / "images"
    label_root = stage / "train" / "labels"
    count = 0
    staged_labels = []

    for root in roots:
        pairs = find_roboflow_pairs(root)
        if not pairs:
            raise ValueError(f"No images/ + labels/ pairs found in {root}")
        for images_dir, labels_dir in pairs:
            for image in iter_images(images_dir):
                relative = image.relative_to(images_dir)
                destination_image = image_root / image.name
                destination_label = label_root / f"{image.stem}.txt"
                if destination_image.exists() or destination_label.exists():
                    raise ValueError(f"Duplicate pooled filename: {image.name}")
                destination_image.parent.mkdir(parents=True, exist_ok=True)
                destination_label.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(image, destination_image)
                source_label = labels_dir / relative.with_suffix(".txt")
                if source_label.is_file():
                    shutil.copy2(source_label, destination_label)
                else:
                    destination_label.write_text("", encoding="utf-8")
                staged_labels.append(destination_label)
                count += 1
    return count, staged_labels

In [ ]:
def convert_annotations(
    input_root: Path,
    output_dataset: Path,
    *,
    class_names: list[str] | dict[int, str] | None = None,
    task: str = "detect",
    validation_fraction: float = 0.2,
    seed: int = 42,
    split_by_video: bool = True,
    deep: bool = False,
    overwrite: bool = False,
):
    """Merge T-REX and Roboflow exports under input_root into one validated
    YOLO dataset with a train/val split, written to output_dataset.

    validation_fraction, seed, and split_by_video control the split ratio;
    see the "Choosing the train/validation split ratio" note above cell 2.
    """
    input_root = input_root.expanduser().resolve()
    output_dataset = output_dataset.expanduser().resolve()
    if not 0 < validation_fraction < 1:
        raise ValueError("validation_fraction must be between 0 and 1")
    if output_dataset.exists() or output_dataset.is_symlink():
        # Refuse to silently reuse or delete an existing output unless the
        # caller explicitly opted in via OVERWRITE_OUTPUT.
        if not overwrite:
            kind = "directory" if output_dataset.is_dir() else "file"
            details = ""
            if output_dataset.is_dir():
                entries = list(output_dataset.iterdir())
                details = f" ({len(entries)} entries)"
            raise FileExistsError(
                f"Output {kind} already exists{details}: {output_dataset}. "
                "Set OVERWRITE_OUTPUT = True to replace it."
            )
        # Guard against wiping the input data (or a parent of it) if the
        # paths were mistakenly configured to overlap.
        if output_dataset == input_root or output_dataset in input_root.parents:
            raise ValueError("Refusing to overwrite INPUT_ROOT or one of its parent directories")
        if output_dataset.parent == output_dataset:
            raise ValueError("Refusing to overwrite a filesystem root")
        print(f"Removing existing output: {output_dataset}")
        if output_dataset.is_dir() and not output_dataset.is_symlink():
            shutil.rmtree(output_dataset)
        else:
            output_dataset.unlink()

    exports = find_exports(input_root)
    roboflow_roots = find_roboflow_roots(input_root)
    if not exports and not roboflow_roots:
        raise ValueError("No *_annotations_yolo or roboflow folders were found")

    print(f"Found {len(exports)} T-REX video exports:")
    for export in exports:
        print(f"  - {export.name}")
    print(f"Found {len(roboflow_roots)} Roboflow folders:")
    for root in roboflow_roots:
        print(f"  - {root}")

    with tempfile.TemporaryDirectory(prefix="trex-yolo-") as temporary:
        stage = Path(temporary) / "merged"
        # Pool every export into one staging folder before splitting, so the
        # split below can draw from all videos/sources at once.
        trex_count, trex_labels = stage_exports(exports, stage) if exports else (0, [])
        roboflow_count, roboflow_labels = stage_roboflow(roboflow_roots, stage)
        image_count = trex_count + roboflow_count
        labels = trex_labels + roboflow_labels
        group_count = len({roboflow_video_id(label.name) for label in labels})
        if split_by_video and group_count < 2:
            raise ValueError("A video-grouped split requires at least two inferred videos")
        if class_names is None:
            class_names = class_names_from_yaml(input_root)
        if class_names is None:
            # No YAML defined names anywhere: fall back to placeholder names
            # derived from the highest class ID actually used in the labels.
            largest = max_class_id(labels)
            class_names = [f"class_{index}" for index in range(max(0, largest) + 1)] or ["object"]
            print(f"No dataset YAML with class names found; generated: {class_names}")

        dataset = Dataset.open(stage, task=task, names=class_names, deep=deep)
        # group_by keeps every frame from the same video together in one
        # split (train or val) instead of shuffling individual frames.
        group_by = (lambda path: roboflow_video_id(path.name)) if split_by_video else None
        planned = dataset.split(
            {"train": 1.0 - validation_fraction, "val": validation_fraction},
            group_by=group_by,
            seed=seed,
            visualize=False,
        )
        result = planned.export(destination=output_dataset, visualize=False)

    print(f"Converted {image_count} frames ({trex_count} T-REX, {roboflow_count} Roboflow).")
    print(f"Dataset: {result.location}")
    print(f"YOLO config: {result.data_yaml}")
    return result

## 4. Build the dataset

Review the paths and settings above, then run this cell.

In [ ]:
result = convert_annotations(
    INPUT_ROOT,
    OUTPUT_DATASET,
    class_names=CLASS_NAMES,
    task=TASK,
    validation_fraction=VALIDATION_FRACTION,
    seed=RANDOM_SEED,
    split_by_video=SPLIT_BY_VIDEO,
    deep=DEEP_DUPLICATE_CHECK,
    overwrite=OVERWRITE_OUTPUT,
)

The output contains standard `train/images`, `train/labels`, `val/images`, `val/labels`, and `data.yaml` paths, together with dataset-fixer validation, lineage, and split-audit reports. The `images` and `labels` directories are flat: files are stored directly inside them, without per-video subfolders.